# 00 · SHAP explain-sample size — stability check

Answers one question: is `N_EXPLAIN=5,000` (`00_SHAP.ipynb` §4) actually large enough for the
mean-|φ| share vector / Simpson index $S$ it feeds into $\kappa_{\mathrm{DiD}}$
(Section~\ref{subsec:shap-did}) to be a stable estimate — and, separately, is it large enough for
region $B$ (fast-tracked, $s>\tau$) alone, since $B$ is a **minority** of a uniform random draw
and is the region $\kappa_{\mathrm{DiD}}$ actually differences.

**Reads only already-saved output — computes no new SHAP values.** `00_SHAP.ipynb` §11 already
writes the full explain-sample φ matrix to the canonical `attributions` parquet
(`<v>_attributions_<split>.parquet`, `ACTIVE_BACKEND`'s result — `interventional` by default,
the backend that feeds every downstream figure). This notebook just re-reads that file through
`loaders.load(...).attributions` and bootstraps it. That is the whole point of it being a
**separate** notebook: re-running `00_SHAP.ipynb` end-to-end to ask this one question would mean
losing its kernel cache and re-doing every SHAP call in it; this one loads one parquet per
(version, split) and finishes in seconds.


### 실행 전 설정

**커널**: analysis `.venv` (`python3`). **선행 조건**: 확인하려는 `(version, split)` 조합마다
`00_SHAP.ipynb`를 그 버전 커널에서 최소 한 번 끝까지 돌려서 `attributions` parquet이 이미
있어야 함 (`ACTIVE_BACKEND`가 기본값 `interventional`인 채로 §11까지 실행된 상태). **바꿀 것은
§1의 `RUNS` 리스트뿐** — 기본값은 baseline v2/v3의 `train` split이 "population 대비 skew가 제일
크고 feature 수도 제일 많아서 가장 불안정하기 쉽다"는 가정에서 나온 최악의 경우 선택이었는데, 이
가정 자체를 검증한 적이 없어서 실제로 쓰이는 `v2/test`, `v3/oot`도 2026-09-27에 추가함 — 넷 다
돌려서 `B_share_pct`/`S_sd`를 직접 비교하면 train이 진짜 제일 나쁜 케이스인지 바로 나옴. 다른 축
(rarity/transport/pnu)이 의심되면 `RUNS`에 더 추가하면 됨.


In [ ]:
# §0 -- setup (analysis .venv kernel)
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from numpy.random import default_rng

ROOT = Path.cwd()
while not (ROOT / "src" / "config.py").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

import config       # noqa: E402
import schema       # noqa: E402
import figstyle     # noqa: E402
from loaders import load  # noqa: E402

figstyle.apply()
figstyle.FIG_DIR = ROOT / "figures" / "real_figures" / "addtional" / "a02_shap_explain_n_stability"
pd.set_option("display.width", 160)
print("ROOT =", ROOT)
print("FIG_DIR =", figstyle.FIG_DIR)


In [ ]:
# §1 -- RUNS to check: (version, split) pairs, each already SHAP-profiled by 00_SHAP.ipynb.
# baseline only -- see the markdown cell above for why per-axis reruns are not needed.
RUNS = [("v2", "train"), ("v3", "train"), ("v2", "test"), ("v3", "oot")]  # +test/oot 2026-09-27: checking the 'train is the worst case' claim directly, not just asserting it

GRID = [100, 250, 500, 1000, 1500, 2000, 3000, 4000, 5000]  # subsample sizes to probe
BOOT = 200    # independent bootstrap draws per grid point (renamed from "B" 2026-09-27 -- too easy to confuse with region B)
SEED = 0


## §2 · Helpers

`_simpson` is exactly Equation~\ref{eq:simpson} ($S=\sum_j p_j^2$) on the mean-$|\phi|$ share
vector of whichever rows it is given. `stability_curve` repeatedly subsamples (without
replacement) `n` rows from a pool of φ rows and recomputes $S$ each time — the spread of those
repeats at a given $n$ is a direct empirical answer to "how much would $S$ have wobbled had the
explain sample been only this big".


In [ ]:
# §2 -- helpers
def _simpson(phi_subset: np.ndarray) -> float:
    """Simpson index S = sum(p_j^2) of the mean-|phi| share vector for these rows."""
    m = np.abs(phi_subset).mean(axis=0)
    p = m / m.sum()
    return float(np.sum(p ** 2))


def stability_curve(phi_pool: np.ndarray, grid: list, b: int = BOOT, seed: int = SEED) -> pd.DataFrame:
    """Bootstrap (without replacement) S at each n in `grid`, b independent draws per n."""
    rng = default_rng(seed)
    n_pool = phi_pool.shape[0]
    rows = []
    for n in sorted(g for g in set(grid) if 0 < g <= n_pool):
        vals = np.array([_simpson(phi_pool[rng.choice(n_pool, size=n, replace=False)])
                          for _ in range(b)])
        rows.append({"n": n, "S_mean": vals.mean(), "S_sd": vals.std(ddof=1),
                     "S_cv_pct": 100 * vals.std(ddof=1) / vals.mean()})
    return pd.DataFrame(rows).set_index("n")


## §3 · Load, split by region, bootstrap, plot

Region membership reuses `loaders.VersionData.decisions` — that version's **own** rule applied to
its **own** scores, the same definition Table~\ref{tab:region-ab} calls $s^{\mathrm{prev}}$ for a
version's own explain sample (that version's own decision is what forced the label on its own
scrapped rows). No new threshold logic, no cross-version join.


In [ ]:
# §3 -- one stability curve pair (whole sample, region B) per RUN
summary_rows = []

for version, split in RUNS:
    print(f"\n=== {version} / {split} ===")
    d = load(version, source="real", split=split)

    attr = d.attributions           # claim_id + one phi column per encoded feature + _base_value
    dec = d.frame[[schema.CLAIM_ID]].assign(above=d.decisions)  # this version's OWN rule, OWN scores
    merged = attr.merge(dec, on=schema.CLAIM_ID, how="inner")
    if len(merged) < len(attr):
        print(f"  {len(attr) - len(merged)} of {len(attr)} explained claim(s) not found in "
              f"{version}/{split}'s targets+scores join -- proceeding with the {len(merged)} "
              f"that matched")

    feature_cols = [c for c in attr.columns if c not in (schema.CLAIM_ID, "_base_value")]
    phi = merged[feature_cols].to_numpy(dtype=float)
    above_mask = merged["above"].to_numpy(dtype=bool)
    n_total, n_B = len(phi), int(above_mask.sum())
    print(f"  explain sample: n={n_total} | region B (this version's own s>tau) inside it: "
          f"n={n_B} ({n_B / n_total:.1%} of the draw)")

    whole_curve = stability_curve(phi, GRID)
    display(whole_curve.round(4))

    if n_B >= 100:
        b_curve = stability_curve(phi[above_mask], GRID + [n_B])
        display(b_curve.round(4))
    else:
        b_curve = None
        print(f"  region B has only {n_B} rows in this draw -- too few for its own curve. The "
              f"whole-sample curve above is an upper bound on how much data buys stability, not "
              f"evidence about region B specifically.")

    fig, ax = plt.subplots(figsize=figstyle.FIG_1)
    ax.errorbar(whole_curve.index, whole_curve["S_mean"], yerr=whole_curve["S_sd"],
                marker="o", label=f"whole sample (max n={n_total})")
    if b_curve is not None:
        ax.errorbar(b_curve.index, b_curve["S_mean"], yerr=b_curve["S_sd"],
                    marker="s", label=f"region B only (max n={n_B})")
    ax.set_xlabel("subsample size n")
    ax.set_ylabel(f"Simpson index S\n(bootstrap mean \u00b1 sd, BOOT={BOOT})")
    ax.set_title(f"{version} \u00b7 {split} \u00b7 {d.attribution_meta.get('backend', '?')} -- "
                 f"explain-sample size stability")
    ax.legend(fontsize=8)
    fig.tight_layout()
    figstyle.save_table(whole_curve, f"{version}_{split}_explain_n_stability_whole")
    if b_curve is not None:
        figstyle.save_table(b_curve, f"{version}_{split}_explain_n_stability_regionB")
    figstyle.save_fig(fig, f"{version}_{split}_explain_n_stability")
    plt.show()

    summary_rows.append({
        "version": version, "split": split, "n_total": n_total, "n_B": n_B,
        "B_share_pct": round(100 * n_B / n_total, 2),
        "S_sd_at_max_n_whole": round(float(whole_curve["S_sd"].iloc[-1]), 5),
        "S_sd_at_max_n_B": round(float(b_curve["S_sd"].iloc[-1]), 5) if b_curve is not None else None,
    })

summary = pd.DataFrame(summary_rows)
print("\n=== summary across RUNS ===")
display(summary)
figstyle.save_table(summary, "explain_n_stability_summary", index=False)


## Notes

- **Region B's share of the draw is read off, not assumed.** `n_B / n_total` above is this
  version's real fast-track rate inside a uniformly random 5,000-row sample, not the population
  fast-track rate of Table~\ref{tab:data-composition} — sampling noise moves it a little, which
  is itself informative about how thin region B's draw is.
- **`S_sd` is the number to read, not `S_mean`.** A flat `S_sd` from some $n$ onward says more
  rows would not have changed the answer; a still-falling `S_sd` at the largest $n$ available
  says the opposite. `S_cv_pct` (coefficient of variation) is the same information normalised by
  scale, useful for comparing region B's curve against the whole-sample curve directly.
- **Encoded features, not aliased.** `phi`'s columns are this version's own encoded feature
  names (`attr` straight off the parquet) — correct for this check (concentration measures never
  collapse encoded features, CLAUDE.md), but never printed or plotted here, so no alias map is
  needed.
- **If this ever needs to run on a mitigated axis**, `loaders.VersionData` has no
  `mitigated_attributions` accessor — read that parquet directly via
  `config.path("mitigated_attributions", version, "real", split=split)` with the axis suffix
  `00_SHAP.ipynb` §11 appends, and reuse `stability_curve` unchanged.


## §4 · Between-draw check (Appendix~\ref{appx:explain-n-stability}) -- is $S$ stable to WHICH 5,000 got drawn?

§3 above (`stability_curve`) only reshuffles the ONE already-drawn 5,000-row attribution
parquet -- it can never show what a genuinely DIFFERENT 5,000-row draw from the same population
would have given, because every resample it runs is a subset of the same fixed rows. This section
answers that harder question instead, and it genuinely needs new SHAP values -- this notebook
still computes none itself, it only reads (this time 5x) what `attribute.py` already wrote.

**Prerequisite -- must be run BEFORE this section, in each version's OWN env (not this analysis
kernel), once per `(version, split, seed)` in `BETWEEN_DRAW_RUNS` x `SEEDS` below:**
```
python attribution/attribute.py --model <that version's model pickle> \
    --features <processed_inputs for (version, split)> \
    --version {version} --split {split} --rows 5000 --seed {seed} \
    --background 500 --backend shap \
    --out src/data/real/detection/shap/{version}/seed_check/{version}_attributions_{split}_seed{seed}.parquet
```
Backend, background size and every other flag held at whatever produced the thesis's own headline
numbers (`interventional`, background 500) -- comparing seeds under different SHAP settings would
put a measurement-comparability confound on top of the sampling question this section asks. Until
all 5 seeds exist for a `(version, split)`, that pair is skipped below with a printed reason
rather than silently omitted.


In [ ]:
# §4a -- setup for the between-draw check
SEEDS = [0, 1, 2, 3, 4]
BETWEEN_DRAW_RUNS = [("v2", "train"), ("v3", "train")]  # same worst-case pair §1 uses
SEED_SUBDIR = "seed_check"
TOPK_BETWEEN_DRAW_FEATURES = 10


def seed_attribution_path(version: str, split: str, seed: int) -> Path:
    """Where the k-th independent-draw attribution parquet lives, alongside the canonical one
    (config.path's own template + SEED_SUBDIR), never a second registered config.py kind for a
    one-off validation exercise."""
    canonical = config.path("attributions", version, "real", split=split)
    return canonical.parent / SEED_SUBDIR / f"{version}_attributions_{split}_seed{seed}.parquet"


In [ ]:
# §4b -- load each found seed, compute S (Equation eq:simpson) per seed, compare
between_summary_rows = []

for version, split in BETWEEN_DRAW_RUNS:
    print(f"\n=== between-draw: {version}/{split} ===")
    seed_S: dict[int, float] = {}
    seed_mabs: dict[int, np.ndarray] = {}
    feature_names = None

    for seed in SEEDS:
        p = seed_attribution_path(version, split, seed)
        if not p.is_file():
            print(f"  seed {seed}: missing ({p}) -- run the attribute.py command above first")
            continue
        attr = pd.read_parquet(p)
        cols = [c for c in attr.columns if c not in (schema.CLAIM_ID, "_base_value")]
        if feature_names is None:
            feature_names = cols
        phi = attr[cols].to_numpy(dtype=float)
        seed_S[seed] = _simpson(phi)
        seed_mabs[seed] = np.abs(phi).mean(axis=0)

    if len(seed_S) < 2:
        print(f"  fewer than 2 of {len(SEEDS)} seeds found for {version}/{split} -- "
              f"nothing to compare yet, skipping")
        continue

    per_seed = pd.DataFrame({"seed": list(seed_S.keys()), "S": list(seed_S.values())})
    display(per_seed)
    figstyle.save_table(per_seed, f"{version}_{split}_between_draw_S_per_seed", index=False)

    s_vals = per_seed["S"].to_numpy()
    s_mean, s_sd = float(s_vals.mean()), float(s_vals.std(ddof=1))
    between_summary_rows.append({
        "version": version, "split": split, "n_seeds_found": len(seed_S),
        "S_mean": round(s_mean, 5), "S_sd": round(s_sd, 5),
        "S_cv_pct": round(100 * s_sd / s_mean, 3),
    })

    # feature-level: top features by the CANONICAL seed (0)'s own ranking, value across all found seeds
    if 0 in seed_mabs:
        top = pd.Series(seed_mabs[0], index=feature_names).sort_values(
            ascending=False).head(TOPK_BETWEEN_DRAW_FEATURES).index
        feat_table = pd.DataFrame(
            {f"seed{seed}": pd.Series(seed_mabs[seed], index=feature_names).loc[top]
             for seed in seed_mabs})
        feat_table.index.name = "feature"
        display(feat_table.round(4))
        figstyle.save_table(feat_table, f"{version}_{split}_between_draw_top_features")
    else:
        print("  seed 0 (the canonical draw) not found -- feature-level table needs it as the "
              "ranking basis, skipped")

between_summary = pd.DataFrame(between_summary_rows)
print("\n=== between-draw summary across RUNS ===")
display(between_summary)
if len(between_summary):
    figstyle.save_table(between_summary, "between_draw_summary", index=False)
